# 05 — LoRA SFT of Gemma with Completion-Only Loss · Google Colab

**Phase 2 / Month 2** · MSc Thesis — ECLIPSE project  
Supervisor: Dr. Panagiotis Kasnesis | Student: Antonios Bastoulis

---

**Setup follows the official Unsloth Gemma-4 tutorial.** The install block (§ 1)
pins the exact versions Gemma 4 expects — `transformers==5.5.0` (unsloth-zoo
hard-caps it at `<=5.5.0`; an unpinned `--upgrade` pulls 5.8.x and the Gemma-4
patch breaks), a `torch`-matched `xformers`, `datasets==4.3.0`, and `timm`. This
keeps training rendering the *same* chat template the model was tuned against.

Completion-only loss is applied with Unsloth's `train_on_responses_only`, which
masks the prompt so cross-entropy is computed only on the assistant turn (the
`<action>` lines). This is the Unsloth-supported replacement for TRL's
`DataCollatorForCompletionOnlyLM` — so **TRL is left unpinned**, exactly as the
tutorial does. Without the mask the ~480-token system prompt would dilute the
~10-20-token action target ~25×, and the model never learns to CHARGE.

Pipeline:
1. Pull the SAC distillation JSONL produced by 04.
2. Load Gemma-4 E4B via **Unsloth** `FastModel` — 4-bit + LoRA.
3. Filter no-op rows (`filter_uninformative_rows`).
4. SFT with TRL's `SFTTrainer` + Unsloth `train_on_responses_only`.
5. Evaluate vs. no-control + RBC on a 1-week mid-dataset slice.


## § 0 — Runtime & Config
> **Edit this cell only.** Nothing else needs changing.

In [ ]:
import os, sys, subprocess, time, warnings, json, random
import numpy as np

# ── Repo + dataset selection ──────────────────────────────────────────────
GITHUB_REPO    = "https://github.com/antonisbast/eclipse-thesis.git"  # adjust if needed
REPO_DIR       = "/content/eclipse-thesis"
DATASET_GLOB   = "notebooks/artifacts/sft_datasets/sac_*.jsonl"  # picks newest match

# ── Model selection ───────────────────────────────────────────────────────
# Unsloth-optimized Gemma 4 E4B
MODEL_ID:       str  = "unsloth/gemma-4-E4B-it"
LOAD_IN_4BIT:   bool = True
MAX_NEW_TOKENS: int  = 80      # action block is ~30-50 tokens; 80 leaves headroom

# ── LoRA / SFT hyperparameters ────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.0    # Unsloth recommends 0 for fastest path
EPOCHS         = 1      # was 3 — loss flattened at step ~200/1866 in v3 run
BATCH_SIZE     = 2
GRAD_ACCUM     = 8      # effective batch 16
LEARNING_RATE  = 5e-5
# Keep model max_seq_length aligned with SFTConfig.max_length so we don't
# allocate activation buffers for tokens we never use. Sample length is
# ~490 tokens; 640 leaves ~30% headroom.
MAX_SEQ_LEN    = 640

# ── Evaluation slice ──────────────────────────────────────────────────────
EVAL_START     = 4000           # mid-summer slice, batteries primed by then
EVAL_LEN       = 168            # 1-week eval; raise to 8760 for full year

# ── CityLearn version ────────────────────────────────────────────────────
# Pin 2.6.0b2 so src.eval.evaluate_v2() works (matches nb 04 / 06).
CITYLEARN_VERSION = "2.6.0b2"

# ── Misc ──────────────────────────────────────────────────────────────────
HF_TOKEN: str = ""
SEED          = 42

try:
    import torch
    if torch.cuda.is_available():
        _gpu  = torch.cuda.get_device_name(0)
        _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ GPU: {_gpu}  ({_vram:.1f} GB VRAM)")
    else:
        print("⚠  No GPU — set Runtime → Change runtime type → T4 GPU")
except ImportError:
    print("torch not yet installed — will be in § 1")


✓ GPU: NVIDIA L4  (23.7 GB VRAM)


## § 1 — Install Dependencies

In [ ]:
# CityLearn 2.6 is a pre-release on PyPI — pin the version explicitly so we
# get the same evaluate_v2() API as the rest of the pipeline (nb 04 / 06).
# --no-deps because CityLearn pulls heavy/legacy deps we don't need at eval
# time (e.g. some EnergyPlus extras). Runtime deps installed explicitly.
!pip install -q numpy gymnasium doe-xstock nrel-pysam
!pip install -q --pre "CityLearn=={CITYLEARN_VERSION}" --no-deps

import citylearn
assert citylearn.__version__.startswith("2.6"), (
    f"Expected CityLearn 2.6.x, got {citylearn.__version__}. "
    f"src.eval depends on evaluate_v2() which only exists in 2.6+."
)
print(f"✓ CityLearn {citylearn.__version__}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 157.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 13.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.6/423.6 kB 26.0 MB/s eta 0:00:00
✓ CityLearn 2.6.0b2


In [ ]:
# Unsloth + Gemma-4 SFT stack — versions pinned to match the OFFICIAL Unsloth
# Gemma-4 tutorial. This matters: Gemma 4 + Unsloth's patch require
# transformers==5.5.0 (unsloth-zoo hard-caps it at <=5.5.0). An unpinned
# `pip install --upgrade transformers` pulls 5.8.x and the Gemma-4 patch breaks.
import os, re, torch

# xformers wheel must match the installed torch — same table the tutorial uses.
_v = re.match(r"[\d]+\.[\d]+", str(torch.__version__)).group(0)
_xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1",
                            "2.8": "0.0.32.post2"}.get(_v, "0.0.34")

!pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {_xformers} peft trl triton unsloth
!pip install -q --no-deps --upgrade "torchao>=0.16.0"
# transformers 5.5.0 + matched tokenizers — installed LAST, --no-deps, so no
# upstream package silently bumps them past Unsloth's supported ceiling.
!pip install -q --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install -q torchcodec
# timm is needed by Gemma 4 — E4B carries vision/audio towers even when we
# only fine-tune the text layers.
!pip install -q --no-deps --upgrade timm

torch._dynamo.config.recompile_limit = 64

import unsloth          # triggers Unsloth's model patching
import trl
# Completion-only loss is applied via Unsloth's train_on_responses_only (§ 6),
# NOT DataCollatorForCompletionOnlyLM — so TRL is left unpinned, like the tutorial.
from unsloth.chat_templates import train_on_responses_only  # noqa: F401
import transformers
print(f"✓ Unsloth + SFT stack installed")
print(f"  transformers=={transformers.__version__}  (expect 5.5.0)  |  trl=={trl.__version__}")
assert transformers.__version__.startswith("5.5"), (
    f"transformers is {transformers.__version__}, expected 5.5.x — "
    f"unsloth-zoo caps it at <=5.5.0; re-run this cell."
)


## § 1.5 — Optional Performance Dep (best-effort)

`cut_cross_entropy` is Unsloth's memory-efficient cross-entropy — with it the
effective batch can grow without OOM. `unsloth-zoo` lists it as an optional
extra (not pulled in by the `--no-deps` install in § 1), so we install it here.
The install is wrapped so a failure does not brick the notebook — Unsloth falls
back to standard CE.

Attention kernels (`xformers`) and `hf_transfer` (parallel HF downloads) were
already installed with matched versions in § 1; this cell just activates
`hf_transfer` and reports what made it in.


In [ ]:
# Optional perf dep — best-effort. A failure here does NOT break the notebook.
import subprocess, sys, importlib, os

# hf_transfer was installed in § 1 — activate the parallel-download path.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# cut_cross_entropy: Unsloth's memory-efficient CE (optional unsloth-zoo extra).
try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "cut_cross_entropy"],
        check=True, timeout=300,
    )
    print("✓ cut_cross_entropy installed")
except Exception as e:
    print(f"✗ cut_cross_entropy ({type(e).__name__}) — Unsloth uses standard CE, fine")

# Report what is actually importable.
print("\nPerf backend availability:")
for mod in ["xformers", "cut_cross_entropy", "hf_transfer"]:
    try:
        importlib.import_module(mod)
        print(f"  ✓ {mod}")
    except Exception as e:
        print(f"  ✗ {mod}  ({type(e).__name__})")


## § 2 — Clone Repository (pulls `src/sft.py` and the JSONL dataset)

In [ ]:
if not os.path.exists(REPO_DIR):
    res = subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR],
                         capture_output=True, text=True)
    print(res.stdout or res.stderr)
else:
    res = subprocess.run(["git", "-C", REPO_DIR, "pull"],
                         capture_output=True, text=True)
    print("Repo present —", res.stdout.strip() or "up to date")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

Cloning into '/content/eclipse-thesis'...



## § 3 — Load Distillation Dataset

In [ ]:
import glob
from pathlib import Path

from src.sft import filter_uninformative_rows

candidates = sorted(glob.glob(os.path.join(REPO_DIR, DATASET_GLOB)))
assert candidates, f"No JSONL matching {DATASET_GLOB} in repo. Run notebook 04 + push first."
JSONL_PATH = candidates[-1]
print(f"Using dataset: {JSONL_PATH}")

raw_rows = [json.loads(l) for l in Path(JSONL_PATH).read_text().strip().split("\n")]
print(f"Raw examples : {len(raw_rows)}")

# Drop rows where EVERY building's action is a physical no-op
# (discharge from SoC≈0, charge into SoC≈100, or |a|<0.05).
# Removes ~30-35% of rows and rebalances the action distribution.
raw_rows = filter_uninformative_rows(raw_rows)
print(f"After filter : {len(raw_rows)}")

print("\nFirst row preview:")
print("  prompt :", raw_rows[0]["prompt"][:200].replace("\n", " | "))
print("  response:", raw_rows[0]["response"][:200].replace("\n", " | "))


Using dataset: /content/eclipse-thesis/notebooks/artifacts/sft_datasets/sac_merlin_distill_20260514_195551_loaded.jsonl
Raw examples : 17518
After filter : 9946

First row preview:
  prompt : STATE: | Month 8, Mon 08:00  |  price=0.220 (LOW)  |  carbon=0.203 (MID) | Buildings: |   B0: SoC=  2.9%  load=0.63 kWh  last_net=-0.28 kWh  solar=HIGH |   B1: SoC=  0.0%  load=1.65 kWh  last_net=+1.12 kWh  s
  response: <action building=0>CHARGE_20</action> | <action building=1>DISCHARGE_20</action> | <action building=2>CHARGE_20</action>


## § 4 — Load Gemma + Attach LoRA (Unsloth `FastModel`)

`FastModel.from_pretrained` returns the model **and** tokenizer with Unsloth's
kernels patched in. `FastModel.get_peft_model` attaches LoRA adapters in
Unsloth-optimized form (no `prepare_model_for_kbit_training` needed).

In [ ]:
from unsloth import FastModel
import torch
import time

# Clear any lingering memory from previous failed runs
torch.cuda.empty_cache()

_t0 = time.time()
model, tokenizer = FastModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,  # Let Unsloth auto-detect optimal precision for Gemma 4
    load_in_4bit    = LOAD_IN_4BIT,
    full_finetuning = False,
    token           = HF_TOKEN or None,
)
print(f"Base model loaded in {time.time()-_t0:.1f}s")

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # text-only distillation
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    random_state   = SEED,
)
model.print_trainable_parameters()

==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.8.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Base model loaded in 87.1s
trainable params: 34,881,536 || all params: 7,975,982,368 || trainable%: 0.4373


## § 5 — Build HF Dataset with Gemma Chat Template

Gemma rejects the `system` role, so the SFT prompt is merged into the user
turn (same workaround as notebook 03's `LocalHFProvider`).

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

from src.sft import make_sft_prompt

# Use Unsloth's curated Gemma-4 chat template so training AND inference render
# the exact template the model was tuned against (parity with the official
# Unsloth Gemma-4 tutorial). Reassigns the global `tokenizer`.
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

N_BUILDINGS   = 3
SYSTEM_PROMPT = make_sft_prompt(N_BUILDINGS)

# Pin padding side — causal LM training requires right-padding so that
# the loss is computed on real tokens, not on left-pad noise. Default
# differs across tokenizer versions, so set it explicitly.
text_tok = getattr(tokenizer, "tokenizer", tokenizer)
text_tok.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_is_gemma = "gemma" in MODEL_ID.lower()

def to_chat(row):
    if _is_gemma:
        msgs = [
            {"role": "user",      "content": f"{SYSTEM_PROMPT}\n\n{row['prompt']}"},
            {"role": "assistant", "content": row["response"]},
        ]
    else:
        msgs = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": row["prompt"]},
            {"role": "assistant", "content": row["response"]},
        ]
    # .removeprefix("<bos>"): the chat template renders a literal <bos>, but
    # SFTTrainer's tokenization prepends another one. Keeping both yields a
    # double-BOS prefix the model never saw in pretraining AND a train/
    # inference mismatch (inference adds only one). Strip it here so the
    # tokenizer re-adds exactly one — matches the Unsloth Gemma-4 tutorial.
    text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=False
    ).removeprefix("<bos>")
    return {"text": text}

# num_proc parallelises chat-template rendering — ~5-10x faster on Colab's 8 vCPUs.
ds = Dataset.from_list(raw_rows).map(
    to_chat,
    remove_columns=["prompt", "response"],
    num_proc=os.cpu_count() or 4,
)
print(f"Dataset built: {len(ds)} examples  |  fields: {ds.column_names}")
print("\nSample formatted text (truncated):")
print(ds[0]["text"][:500])
print("...")

# ── 95/5 train/eval split for overfitting detection + early stopping ─────
ds_split = ds.train_test_split(test_size=0.05, seed=SEED)
train_ds, eval_ds = ds_split["train"], ds_split["test"]
print(f"\nSplit: train={len(train_ds)}  eval={len(eval_ds)}")


In [ ]:
# Quick sanity: Unsloth's completion-only masking helper is importable.
from unsloth.chat_templates import train_on_responses_only
import trl
print(f"trl=={trl.__version__}")
print("train_on_responses_only ready — completion-only loss via Unsloth")


In [ ]:
from google.colab import drive

# This will prompt you to authorize Google Drive access
drive.mount('/content/drive')

Mounted at /content/drive


## § 6 — Train (TRL `SFTTrainer` on Unsloth model)

Unsloth's patched model + tokenizer plug straight into TRL's `SFTTrainer`.
Completion-only loss is applied via Unsloth's `train_on_responses_only`,
which masks the prompt so cross-entropy is computed only on the assistant
turn (the `<action>` lines) — the Unsloth-supported replacement for TRL's
`DataCollatorForCompletionOnlyLM` (which needed brittle `tokenizer.tokenizer`
hacks for the Gemma-4 multimodal processor).

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import time

# Save directly to Drive so a Colab disconnect doesn't nuke the run.
# Drive must be mounted (cell 15) BEFORE this cell.
OUT_DIR = f"/content/drive/MyDrive/eclipse-thesis/sft_out_v5/{time.strftime('%Y%m%d_%H%M%S')}"

sft_cfg = SFTConfig(
    output_dir                  = OUT_DIR,
    num_train_epochs            = 1,            # was 3 — loss flattened at step ~200 of 1866
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    max_length                  = 640,
    logging_steps               = 25,

    # ── Checkpointing → Drive (survives disconnects) ─────────────────────
    save_strategy               = "steps",
    save_steps                  = 100,
    save_total_limit            = 3,            # rotate, don't fill Drive

    # ── Eval split — detect overfitting, enable early stopping ───────────
    eval_strategy               = "steps",
    eval_steps                  = 100,
    per_device_eval_batch_size  = BATCH_SIZE * 4,  # no grad memory at eval time
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,

    bf16                        = True,
    fp16                        = False,
    optim                       = "paged_adamw_8bit",
    warmup_ratio                = 0.03,
    lr_scheduler_type           = "linear",
    report_to                   = "none",
    seed                        = SEED,
    dataset_text_field          = "text",
    # Required on L4 (22GB) — Gemma-4 E4B + 4-bit + LoRA + bs 2 OOMs without it.
    # If you move to A100 (80GB), set both to False for ~30% speedup.
    gradient_checkpointing         = True,
    gradient_checkpointing_kwargs  = {"use_reentrant": False},
)

trainer = SFTTrainer(
    model           = model,
    args            = sft_cfg,
    train_dataset   = train_ds,
    eval_dataset    = eval_ds,
    processing_class= tokenizer,
)

# Completion-only loss via Unsloth's train_on_responses_only: masks everything
# up to and including the assistant-turn marker, so cross-entropy is computed
# only on the <action ...> tokens. The prompt is ~480 tokens of templated
# boilerplate, the response is ~10-20 action tokens — without this mask,
# >95% of the gradient signal would teach the model to autocomplete its own
# system prompt. Markers match the Unsloth Gemma-4 chat template (§ 5).
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part    = "<|turn>model\n",
)

# ── Verify masking + single <bos> BEFORE the ~4h train ──────────────────
_tt    = getattr(tokenizer, "tokenizer", tokenizer)
_ex    = trainer.train_dataset[0]
_n_bos = sum(t == _tt.bos_token_id for t in _ex["input_ids"])
_sup   = [t for t, l in zip(_ex["input_ids"], _ex["labels"]) if l != -100]
print(f"sanity | <bos> count        = {_n_bos}  (expect 1)")
print(f"sanity | supervised tokens  = {len(_sup)}/{len(_ex['labels'])}")
print(f"sanity | supervised span    = {_tt.decode(_sup)!r}")
assert _n_bos == 1, (
    f"Expected exactly one <bos>, got {_n_bos} — check .removeprefix('<bos>') in § 5"
)
assert 0 < len(_sup) < len(_ex["labels"]), (
    "Masking failed — completion-only loss not applied (supervised span empty or full)"
)

_t0 = time.time()
trainer.train()
print(f"\nSFT done in {time.time()-_t0:.1f}s")

# Save the best model (load_best_model_at_end=True already loaded it).
# OUT_DIR is on Drive, so this also survives runtime shutdown.
ADAPTER_DIR = f"{OUT_DIR}/lora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved: {ADAPTER_DIR}")


## § 12 — Persist Adapter to Drive (optional)

In [ ]:
# Ensure the target directory exists
!mkdir -p /content/drive/MyDrive/eclipse-thesis/sft_adaptersV5/

# Copy the saved adapter to your Drive
!cp -r {ADAPTER_DIR} /content/drive/MyDrive/eclipse-thesis/sft_adaptersV5/

print(f"\nAdapter successfully backed up to Google Drive!")
print(f"Look for it in: MyDrive/eclipse-thesis/sft_adapters/")


Adapter successfully backed up to Google Drive!
Look for it in: MyDrive/eclipse-thesis/sft_adapters/


## § 7 — Build Inference Provider (Unsloth fast inference)

`FastModel.for_inference(model)` enables Unsloth's 2× faster inference path
(disables training-only autograd hooks). Same `.complete()` / `.step()` API
as notebook 03's `LocalHFProvider`, so the rollout helper below is unchanged.

In [ ]:
import logging, re
from src.sft import parse_actions, make_sft_prompt

_logger = logging.getLogger("sft_slm")
# Opening-tag-only regex — DIFFERENT from src.agent.ACTION_RE (which captures the
# full <action ...>TOKEN</action> for parsing). This one is used elsewhere for
# a quick "did the model emit any action tags?" check.
_ACTION_TAG_RE = re.compile(r"<action\s+building\s*=\s*\d+\s*>", re.IGNORECASE)

FastModel.for_inference(trainer.model)  # 2× faster inference


class SFTProvider:
    """Inference provider wrapping the LoRA-fine-tuned Unsloth model."""

    def __init__(self, model, tokenizer, max_new_tokens: int = 200):
        self.model         = model.eval()
        self.tokenizer     = tokenizer
        self.max_new_tokens= max_new_tokens
        self.label         = f"sft:{MODEL_ID.split('/')[-1]}"
        self._is_gemma     = "gemma" in MODEL_ID.lower()
        self._device       = next(model.parameters()).device

    def complete(self, system: str, user: str, max_tokens=None) -> str:
        max_new = max_tokens or self.max_new_tokens
        if self._is_gemma:
            msgs = [{"role": "user", "content": f"{system}\n\n{user}"}]
        else:
            msgs = [{"role": "system", "content": system},
                    {"role": "user",   "content": user}]
        enc = self.tokenizer.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True,
            return_tensors="pt", return_dict=True,
        ).to(self._device)
        with torch.no_grad():
            out = self.model.generate(
                **enc,
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
            )
        new_tokens = out[0][enc["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def step(self, state_text: str, system=None, n_buildings: int = 6,
             max_retries: int = 2, **kw):
        sys_p = system or make_sft_prompt(n_buildings)
        last  = ""
        for _ in range(max_retries):
            last = self.complete(sys_p, f"STATE:\n{state_text}")
            if _ACTION_TAG_RE.search(last):
                return parse_actions(last, n_buildings), last, False
        return [0.0]*n_buildings, last, True


slm = SFTProvider(trainer.model, tokenizer, max_new_tokens=MAX_NEW_TOKENS)
print(f"Provider ready: {slm.label}")


Provider ready: sft:gemma-4-E4B-it


## § 8 — Evaluation Environment Factory (Colab)

In [ ]:
import citylearn
from citylearn.citylearn import CityLearnEnv
from citylearn.agents.rbc import BasicBatteryRBC
from src.env import (
    make_env, snapshot_state, MERLINReward,
    OBSERVATIONS, ACTIVE_ACTIONS,
    TRAINING_BUILDINGS, HELDOUT_BUILDINGS, BUILDINGS,
)
from src.eval import evaluate, comparison_table
from src.sft  import render_state

warnings.filterwarnings("ignore")
np.random.seed(SEED); random.seed(SEED)

# Default to the 3-building TRAINING slice — this matches N_BUILDINGS=3 in
# cell 15 and the slice geometry the JSONL was emitted on. Previously this
# defaulted to BUILDINGS=[0..5], which made the headline eval below a 6-bldg
# OOD rollout that the SLM had never been trained for.
#
# Schema source: use the project-wide src.env.make_env so this notebook hits
# the SAME local-path schema as nb 04 (dataset dump) — the repo was cloned
# in § 2 so the data/ tree is on disk. Previously this notebook constructed
# CityLearnEnv directly with `schema="citylearn_challenge_2022_phase_all"`,
# which triggers a separate download from the pinned CityLearn tag and may
# drift from the file nb 04 used.
def make_colab_env(start=0, end=8759, buildings=None) -> CityLearnEnv:
    return make_env(
        buildings = buildings if buildings is not None else TRAINING_BUILDINGS,
        start     = start,
        end       = end,
        reward_fn = "merlin",
    )

print(f"CityLearn {citylearn.__version__}  |  default eval buildings = TRAINING_BUILDINGS = {TRAINING_BUILDINGS}")


CityLearn 2.6.0b2  |  default eval buildings = TRAINING_BUILDINGS = [0, 1, 2]


## § 9 — Rollout Helpers

In [ ]:
def run_slm_policy(name: str, provider, start=EVAL_START, length=EVAL_LEN,
                   n_buildings: int = N_BUILDINGS, verbose_every: int = 24):
    env = make_colab_env(start=start, end=start+length-1)
    env.reset()
    sys_p = make_sft_prompt(n_buildings)
    done, t, t0, n_fb = False, 0, time.time(), 0
    while not done:
        snap = snapshot_state(env)
        state_text = render_state(snap)
        acts, raw, fb = provider.step(state_text, system=sys_p, n_buildings=n_buildings)
        n_fb += int(fb)
        _, _, term, trunc, _ = env.step([[float(a)] for a in acts])
        done = bool(term or trunc)
        if verbose_every and (t % verbose_every == 0):
            elapsed = time.time() - t0
            print(f"  t={t:4d} | {elapsed:.0f}s | fallbacks={n_fb}")
        t += 1
    print(f"[{name}] {t} steps in {time.time()-t0:.1f}s | fallbacks={n_fb}/{t}")
    return env


def run_rbc(start=EVAL_START, length=EVAL_LEN):
    env = make_colab_env(start=start, end=start+length-1)
    BasicBatteryRBC(env=env).learn(episodes=1)
    return env


def run_noop(start=EVAL_START, length=EVAL_LEN):
    env = make_colab_env(start=start, end=start+length-1)
    env.reset()
    n_b = len(env.buildings)
    done = False
    while not done:
        _, _, term, trunc, _ = env.step([[0.0] for _ in range(n_b)])
        done = bool(term or trunc)
    return env

## § 10 — Run Baselines + SFT-SLM

In [ ]:
print("── No-control baseline ──")
env_noop = run_noop()

print("\n── RBC ──")
env_rbc  = run_rbc()

print(f"\n── SFT SLM ({slm.label}) ──")
env_slm  = run_slm_policy("sft_slm", slm)

── No-control baseline ──

── RBC ──

── SFT SLM (sft:gemma-4-E4B-it) ──
  t=   0 | 50s | fallbacks=0


## § 11 — Evaluation

In [ ]:
# Challenge KPIs via src.eval (evaluate_v2 — CityLearn 2.6+)
eval_results = [
    evaluate(env_noop, label="No-Control"),
    evaluate(env_rbc,  label="RBC"),
    evaluate(env_slm,  label=slm.label),
]
challenge_df, zne_df = comparison_table(eval_results)
print("Challenge scores — lower is better; No-Control is the reference baseline.")
display(challenge_df.round(4))

print("\nZNE + self-consumption:")
display(zne_df[["solar generation (kWh)", "grid import (kWh)",
                "ZNE ratio (solar / import)", "self-consumption ratio"]].round(4))


## § 13 — Battery State of Charge (SOC) Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_buildings = len(env_slm.buildings)
fig, axes = plt.subplots(nrows=n_buildings, ncols=1, figsize=(12, 2 * n_buildings), sharex=True)

if n_buildings == 1:
    axes = [axes]

for i, b in enumerate(env_slm.buildings):
    # Extract the SOC history for the episode
    soc_history = b.electrical_storage.soc

    axes[i].plot(soc_history, label=f'Building {i}', color='tab:blue', linewidth=1.5)
    axes[i].set_ylabel('SOC')
    axes[i].set_ylim(-0.05, 1.05)  # SOC is strictly between 0 and 1
    axes[i].legend(loc='upper right')
    axes[i].grid(True, linestyle='--', alpha=0.7)

axes[-1].set_xlabel('Time Step (Hours)')
fig.suptitle(f'Battery State of Charge (SOC) over 1 Week - {slm.label}', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

## § 14 — Debugging: Inspect Raw SLM Output
Let's run a single step to see exactly what the model is generating.

In [ ]:
from src.env import snapshot_state
from src.sft import render_state, make_sft_prompt

# Create a test environment and get the first state
env_test = make_colab_env(start=EVAL_START, end=EVAL_START+5)
env_test.reset()
snap = snapshot_state(env_test)
state_text = render_state(snap)
sys_p = make_sft_prompt(N_BUILDINGS)

print("=== RAW SLM GENERATION TEST ===\n")
print("--- PROMPT STATE ---")
print(state_text[:500] + "\n... [truncated]")

print("\n--- MODEL RESPONSE ---")
# Run the complete method to get the raw string
raw_resp = slm.complete(sys_p, f"STATE:\n{state_text}")
print(raw_resp)

print("\n--- PARSED ACTIONS ---")
acts, raw, fallback = slm.step(state_text, system=sys_p, n_buildings=N_BUILDINGS)
print(f"Parsed actions: {acts}")
print(f"Did it trigger a fallback (parsing error)? {fallback}")

## § 15 — 1-Day Inference Trace
Run a 24-hour rollout and print the exact states and responses for inspection.

In [ ]:
print("=== 1-DAY INFERENCE TRACE ===")
# 24 steps for a single day
env_day = make_colab_env(start=EVAL_START, end=EVAL_START+23)
env_day.reset()
sys_p = make_sft_prompt(N_BUILDINGS)

done = False
step_idx = 0

while not done:
    print(f"\n{'='*40}")
    print(f" STEP {step_idx:02d}")
    print(f"{'='*40}")

    snap = snapshot_state(env_day)
    state_text = render_state(snap)

    print("[INPUT STATE]")
    # Printing the first few lines to keep output readable,
    # but we can print the whole thing if preferred.
    lines = state_text.split('\n')
    print('\n'.join(lines[:4]))
    print("  ... [buildings truncated] ...")

    acts, raw_resp, fallback = slm.step(state_text, system=sys_p, n_buildings=N_BUILDINGS)

    print("\n[MODEL OUTPUT]")
    print(raw_resp.strip())
    print(f"\n-> Parsed Actions: {acts} (Fallback: {fallback})")

    _, _, term, trunc, _ = env_day.step([[float(a)] for a in acts])
    done = bool(term or trunc)
    step_idx += 1


## § 16 — Inspect the Exact Inference Prompt
Print the fully formatted prompt (with chat template tokens) sent to the model.

In [ ]:
from src.env import snapshot_state
from src.sft import render_state, make_sft_prompt

# Get step 0 state
env_test = make_colab_env(start=EVAL_START, end=EVAL_START+1)
env_test.reset()
snap = snapshot_state(env_test)
state_text = render_state(snap)

sys_p = make_sft_prompt(N_BUILDINGS)
user_p = f"STATE:\n{state_text}"

# Combine them exactly as SFTProvider.complete() does for Gemma
msgs = [{"role": "user", "content": f"{sys_p}\n\n{user_p}"}]

# Apply the chat template to see the raw input string
raw_prompt = slm.tokenizer.apply_chat_template(
    msgs,
    tokenize=False,
    add_generation_prompt=True
)

print("=== EXACT MODEL INPUT PROMPT ===\n")
print(raw_prompt)

## § 17 — Inference with CoT Minimal Prompt
Testing the modified prompt that introduces Chain-of-Thought `<thought>` blocks and specific operational constraints.

In [ ]:
# CoT prompt for the optional CoT-eval section comes from src.agent
# (canonical CoT prompt). The fine-tuned SLM was trained on make_sft_prompt
# (no <thought> block), so applying make_minimal_prompt at eval time is OOD —
# this section exists to QUANTIFY that drift, not to be used as the headline KPI.
from src.agent import make_minimal_prompt
print('CoT prompt for OOD comparison:')
print(make_minimal_prompt(N_BUILDINGS))

## § 18 — Verify CoT Prompt Format
Let's print the exact chat-formatted string that gets sent to the model during the CoT inference to confirm the state is included.

In [ ]:
# Combine the CoT system prompt and the current state
msgs = [{"role": "user", "content": f"{sys_p_cot}\n\nSTATE:\n{state_text}"}]

# Apply the Gemma chat template
raw_cot_prompt = slm.tokenizer.apply_chat_template(
    msgs,
    tokenize=False,
    add_generation_prompt=True
)

print("=== EXACT CoT MODEL INPUT PROMPT ===\n")
print(raw_cot_prompt)

## § 19 — 1-Day Evaluation with CoT Prompt
Let's run a 24-hour episode using the CoT prompt to see if the reasoning step improves battery utilization.

In [ ]:
import time
from src.env import snapshot_state
from src.sft import render_state
import pandas as pd
import matplotlib.pyplot as plt

print("=== 1-DAY CoT EVALUATION ===")
# 24 steps = 1 day
env_cot = make_colab_env(start=EVAL_START, end=EVAL_START+23)
env_cot.reset()

done, t, t0, n_fb = False, 0, time.time(), 0

while not done:
    snap = snapshot_state(env_cot)
    state_text = render_state(snap)

    # Use the CoT system prompt and allow more tokens for the thought block
    acts, raw, fb = slm.step(state_text, system=sys_p_cot, n_buildings=N_BUILDINGS, max_tokens=250)

    n_fb += int(fb)
    _, _, term, trunc, _ = env_cot.step([[float(a)] for a in acts])
    done = bool(term or trunc)

    if t % 6 == 0:
        print(f"  t={t:2d} | fallbacks={n_fb}")
    t += 1

print(f"Done in {time.time()-t0:.1f}s | fallbacks={n_fb}/{t}")

# Evaluate KPIs via src.eval (evaluate_v2)
res_cot = evaluate(env_cot, label='CoT (1 Day)')
display(res_cot.challenge.round(4))
display(res_cot.zne[['solar generation (kWh)', 'grid import (kWh)',
                     'ZNE ratio (solar / import)', 'self-consumption ratio']].round(4))

# Plot SOC
fig, axes = plt.subplots(nrows=N_BUILDINGS, ncols=1, figsize=(12, 2 * N_BUILDINGS), sharex=True)
if N_BUILDINGS == 1: axes = [axes]

for i, b in enumerate(env_cot.buildings):
    axes[i].plot(b.electrical_storage.soc, label=f'Building {i}', color='tab:orange', linewidth=2)
    axes[i].set_ylabel('SOC')
    axes[i].set_ylim(-0.05, 1.05)
    axes[i].legend(loc='upper right')
    axes[i].grid(True, linestyle='--', alpha=0.7)

axes[-1].set_xlabel('Time Step (Hours)')
fig.suptitle('Battery SOC over 1 Day - CoT Prompt', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()
